# Tome 5, Chapter 12: Modular forms (Python companion)

Exercises first, solutions at the end. We fold the upper half-plane into the fundamental domain (Theorem 12.2), test the transformation laws of the Eisenstein series (Theorem 12.6) and of the eta function (Theorem 12.7), check the divisor identities that fall out of the dimension formula (Corollary 12.10, Exercise 12.1), compute twenty thousand values of Ramanujan's $\tau$ (Corollary 12.11, Theorems 12.13 and 12.14), look at the Sato–Tate law, and finally find short vectors in planar lattices with the same algorithm that reduces a point to $F$. Requirements: numpy, sympy.

*Before each solution, predict what the output will look like.*

## Exercises

**P12.1 (★) Folding the half-plane.** Write a function that moves a point $\tau$ of the upper half-plane into $F$ by translations $T^m$ and inversions $S$, as in the proof of Theorem 12.2. Test it on points very close to the real axis. How many inversions are needed?

**P12.2 (★) Eisenstein series.** From the $q$-expansions of Theorem 12.6, check numerically that $E_k(-1/\tau)=\tau^kE_k(\tau)$ for $k=4,6$, and that $E_6(i)=0$ and $E_4(\rho)=0$ (Exercise 12.3).

**P12.3 (★) Divisor identities.** Check Corollary 12.10 and the identity of Exercise 12.1 for all $n\le500$.

**P12.4 (★★) Ramanujan's tau.** Compute $\tau(n)$ for $n\le20000$. Expanding $\prod(1-q^n)^{24}$ directly is slow; use Jacobi's identity $\prod(1-q^n)^3=\sum_{k\ge0}(-1)^k(2k+1)q^{k(k+1)/2}$ and raise a sparse series to the $8$th power. Check multiplicativity, the recursion of Theorem 12.14, the congruence modulo $691$ of Corollary 12.11, and Deligne's bound $|\tau(p)|\le2p^{11/2}$.

**P12.5 (★) Sato–Tate.** With $x_p=\frac{\tau(p)}{2p^{11/2}}$, compare the moments of $x_p^2,x_p^4,x_p^6$ with those of the semicircle law, $\frac14,\frac18,\frac5{64}$. Does $\tau(n)$ vanish for some $n\le20000$ (Lehmer's question)?

**P12.6 (★★) Eta and Hecke's bound.** Check $\eta(-1/\tau)=\sqrt{-i\tau}\,\eta(\tau)$ numerically, and compare $\max|\tau(n)|/n^6$ (Exercise 12.6) with $\max|\tau(n)|/n^{11/2}$.

**P12.7 (★★) CS application: lattice reduction.** A basis $u,v$ of a lattice in $\mathbb R^2=\mathbb C$ is *reduced* if $|u|\le|v|$ and $|\mathrm{Re}(v\bar u)|\le\frac12|u|^2$. Write the Lagrange–Gauss algorithm (subtract the nearest multiple of $u$ from $v$, swap, repeat). Hide a nice basis behind a matrix of $SL_2(\mathbb Z)$ with entries near $10^6$, recover the shortest vector, and compare $v/u$ with what P12.1 gives.

## Solutions

### P12.1

In [ ]:
import math, cmath, random
import numpy as np
from sympy import divisor_sigma, primerange

def reduce_to_F(tau):
    # apply T^m and S until tau lies in F; returns the reduced point and the number of S steps
    steps = 0
    while True:
        tau = complex(tau.real - round(tau.real), tau.imag)
        if abs(tau) >= 1 - 1e-12: return tau, steps
        tau = -1 / tau; steps += 1
random.seed(1)
for _ in range(5):
    t = complex(random.uniform(-3, 3), 10**random.uniform(-4, 0))
    r, k = reduce_to_F(t)
    print(f"tau = {t.real:+.4f}{t.imag:+.6f}i  ->  {r.real:+.4f}{r.imag:+.4f}i in F after {k} inversions")

Each loop is one step of the proof of Theorem 12.2: a translation brings $\mathrm{Re}\,\tau$ into $[-\frac12,\frac12]$, and if $|\tau|<1$ the inversion $S$ strictly increases the imaginary part. Even from $\mathrm{Im}\,\tau\approx10^{-4}$, four inversions suffice: the algorithm is the continued fraction algorithm in disguise (a point $x+iy$ with tiny $y$ follows the continued fraction of $x$, Tome 1), so it needs about $\log\frac1y$ steps.

### P12.2

In [ ]:
def E(k, tau, terms=200):
    coef = {4: 240, 6: -504, 8: 480, 10: -264}[k]
    q = cmath.exp(2j * math.pi * tau)
    return 1 + coef * sum(int(divisor_sigma(n, k - 1)) * q**n for n in range(1, terms))
for tau in (0.3 + 1.1j, -0.2 + 0.9j, 0.05 + 1.5j):
    for k in (4, 6):
        lhs = E(k, -1 / tau); rhs = tau**k * E(k, tau)
        print(f"k={k}, tau={tau}: E_k(-1/tau) = {lhs:.10f},  tau^k E_k(tau) = {rhs:.10f}")
print("E_6(i) =", abs(E(6, 1j)), "  E_4(rho) =", abs(E(4, cmath.exp(2j * math.pi / 3))))

The two sides agree to ten digits: two hundred terms are plenty, since $|q|<0.016$ at every point used here. The zeros come out of order $10^{-15}$, rounding error, exactly as Exercise 12.3 predicts: $i$ is fixed by $S$, so $E_6(i)=i^6E_6(i)=-E_6(i)$, and $\rho$ is fixed by $ST$, which kills $E_4$ in the same way.

### P12.3

In [ ]:
M = 500
s = {k: [0] + [int(divisor_sigma(n, k)) for n in range(1, M + 1)] for k in (3, 5, 7, 9, 11)}
ok7 = all(s[7][n] == s[3][n] + 120 * sum(s[3][m] * s[3][n - m] for m in range(1, n)) for n in range(1, M + 1))
ok9 = all(11 * s[9][n] == 21 * s[5][n] - 10 * s[3][n] + 5040 * sum(s[3][m] * s[5][n - m] for m in range(1, n)) for n in range(1, M + 1))
print(f"sigma_7 identity for n <= {M}: {ok7};  sigma_9 identity: {ok9}")

Both identities hold exactly. Nothing in elementary number theory explains why the convolution $\sum\sigma_3(m)\sigma_3(n-m)$ should be a combination of $\sigma_7(n)$ and $\sigma_3(n)$; the reason is that $M_8$ is one-dimensional, so $E_4^2=E_8$ (Theorem 12.9).

### P12.4

In [ ]:
def tau_table(N):
    # prod(1-q^n)^24 = (sum_k (-1)^k (2k+1) q^{k(k+1)/2})^8, by Jacobi's identity
    e3 = [0] * (N + 1); k = 0
    while k * (k + 1) // 2 <= N:
        e3[k * (k + 1) // 2] = (-1)**k * (2 * k + 1); k += 1
    sp = [(i, c) for i, c in enumerate(e3) if c]
    cur = e3[:]
    for _ in range(7):
        new = [0] * (N + 1)
        for i, c in sp:
            for j in range(N + 1 - i):
                if cur[j]: new[i + j] += c * cur[j]
        cur = new
    return [0] + cur[:N]
N = 20000
tau = tau_table(N)
print("tau(1..12):", tau[1:13])
print("multiplicative on coprime pairs below 200:", all(tau[m * n] == tau[m] * tau[n] for m in range(1, 200) for n in range(1, 200) if math.gcd(m, n) == 1 and m * n <= N))
print("Hecke recursion for p=2,3,5,7:", all(tau[p**(r + 1)] == tau[p] * tau[p**r] - p**11 * tau[p**(r - 1)] for p in (2, 3, 5, 7) for r in range(1, 5) if p**(r + 1) <= N))
print("691 congruence for n <= 2000:", all((tau[n] - s11) % 691 == 0 for n, s11 in ((n, int(divisor_sigma(n, 11))) for n in range(1, 2001))))
P = list(primerange(2, N + 1))
x = np.array([tau[p] / (2 * p**5.5) for p in P])
print(f"Deligne's bound: max |tau(p)| / (2 p^5.5) over {len(P)} primes = {np.abs(x).max():.4f}")

Everything Ramanujan conjectured in 1916 is there: $\tau(1),\dots,\tau(12)$ start $1,-24,252,-1472,4830$, multiplicativity and the Hecke recursion hold on every case tested (Theorem 12.14), and $\tau(n)\equiv\sigma_{11}(n)\pmod{691}$ for all $n\le2000$. The largest $|x_p|$ over the $2262$ primes below $20000$ is $0.987$: Deligne's bound is never violated, and it is nearly attained. The Jacobi trick makes the computation fast: the series $\prod(1-q^n)^3$ has only about $200$ nonzero terms up to $q^{20000}$.

### P12.5

In [ ]:
print(f"moments of x_p: E x^2 = {np.mean(x**2):.4f} (Sato-Tate 1/4),  E x^4 = {np.mean(x**4):.4f} (1/8),  E x^6 = {np.mean(x**6):.4f} (5/64 = {5/64:.4f})")
print("tau(n) = 0 for some n <= 20000?", any(t == 0 for t in tau[1:]))
print("sign of tau(p): proportion positive =", np.mean(x > 0).round(4))

The moments $0.246$, $0.123$, $0.076$ match the semicircle's $\frac14,\frac18,\frac5{64}$ to within two percent (Figure 12.2 shows the histogram). This is the Sato–Tate law for $\Delta$, proved in 2011 by Barnet-Lamb, Geraghty, Harris and Taylor. Signs are a fair coin: $50.35\%$ positive. And $\tau(n)\ne0$ for all $n\le20000$; Lehmer's question is open, but it is known that $\tau(n)\ne0$ for $n$ up to about $8\cdot10^{23}$ (Derickx, van Hoeij and Zeng, 2013).

### P12.6

In [ ]:
def eta(tau, terms=400):
    q = cmath.exp(2j * math.pi * tau)
    pr = 1
    for n in range(1, terms): pr *= 1 - q**n
    return cmath.exp(1j * math.pi * tau / 12) * pr
for t in (0.4 + 0.8j, -0.3 + 1.2j, 0.1 + 0.6j):
    print(f"tau={t}: eta(-1/tau) = {eta(-1 / t):.10f},  sqrt(-i tau) eta(tau) = {cmath.sqrt(-1j * t) * eta(t):.10f}")
r6 = max(abs(tau[n]) / n**6 for n in range(1, N + 1)); r55 = max(abs(tau[n]) / n**5.5 for n in range(1, N + 1))
print(f"max |tau(n)|/n^6 = {r6:.4f},  max |tau(n)|/n^5.5 = {r55:.4f}")

Dedekind's transformation law holds to ten digits. The ratio $|\tau(n)|/n^6$ is largest at $n=1$, where it is $1$, and then decreases: Hecke's bound $O(n^6)$ of Exercise 12.6 is true but wasteful. The ratio $|\tau(n)|/n^{11/2}$ stays bounded by $3.3$ in our range, but it is \emph{not} bounded in general: Deligne's theorem gives $|\tau(n)|\le d(n)n^{11/2}$, and the divisor function $d(n)$ is unbounded, so the right exponent is $\frac{11}2+\varepsilon$.

### P12.7

In [ ]:
def gauss_reduce(u, v):
    # Lagrange-Gauss reduction of a basis of a lattice in R^2 (as complex numbers); returns a reduced basis
    if abs(u) > abs(v): u, v = v, u
    steps = 0
    while True:
        m = round((v * u.conjugate()).real / abs(u)**2)
        v = v - m * u; steps += 1
        if abs(v) >= abs(u): return u, v, steps
        u, v = v, u
random.seed(3)
for _ in range(5):
    # a nice basis, hidden by a random matrix of SL2(Z) with large entries
    u0 = complex(random.randint(50, 100), random.randint(-20, 20)); v0 = u0 * complex(random.uniform(-0.5, 0.5), random.uniform(1, 2))
    a, b = random.randint(10**5, 10**6), random.randint(10**5, 10**6)
    while math.gcd(a, b) != 1: b += 1
    # solve a*d - b*c = 1 with the extended Euclidean algorithm
    g, x0, y0 = a, 1, 0; r, x1, y1 = b, 0, 1
    while r: qq = g // r; g, r = r, g - qq * r; x0, x1 = x1, x0 - qq * x1; y0, y1 = y1, y0 - qq * y1
    d, c = x0, -y0
    u, v = a * u0 + b * v0, c * u0 + d * v0
    ru, rv, k = gauss_reduce(u, v)
    best = min(abs(i * ru + j * rv) for i in range(-3, 4) for j in range(-3, 4) if (i, j) != (0, 0))
    t = rv / ru if (rv / ru).imag > 0 else -rv / ru
    print(f"|u| = {abs(u):.3g}, |v| = {abs(v):.3g}: shortest vector {abs(ru):.2f} (brute force {best:.2f}, hidden {min(abs(u0), abs(v0)):.2f}) after {k} steps; v/u = {t:.4f}, reduce_to_F gives {reduce_to_F(v0 / u0)[0]:.4f}")

In each case the algorithm finds, in five to ten steps, a vector as short as the hidden one, and brute force over small combinations confirms that nothing shorter exists. The ratio $v/u$ of the reduced basis is the point of $F$ that P12.1 finds for $v_0/u_0$: lattice reduction in the plane \emph{is} the reduction of $\tau$ to the fundamental domain, written with vectors instead of Möbius maps. Its higher-dimensional heir, the LLL algorithm (Lenstra, Lenstra, Lovász, 1982), breaks knapsack cryptosystems and weak RSA keys, finds integer relations between real constants, and is the basic tool of lattice-based cryptography, the leading candidate to survive quantum computers.